In [2]:
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np
import faiss
import pickle
from sklearn.preprocessing import normalize


In [3]:
raw_data = pd.read_csv("fra_perfumes.csv")

# Loading Embeddings

In [4]:
with open('data.pkl', 'rb') as f:
   data = pickle.load(f)

In [5]:
desc_embs = np.load("desc_embs.npy")
accord_embs = np.load("accord_embs.npy")
gender_embs = np.load("gender_embss.npy")
brand_embs = np.load("brand_embs.npy")

# Generating Final Embeddings

In [6]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [7]:
W_DESC   = 0.35
W_ACCORD = 0.25
W_GENDER = 0.30
W_BRAND = 0.1

final_embs = (
    W_DESC   * desc_embs +
    W_ACCORD * accord_embs +
    W_GENDER * gender_embs + 
    W_BRAND  * brand_embs
)

final_embs = normalize(final_embs)

dim = final_embs.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(final_embs.astype(np.float32))

In [8]:
brands = sorted(data["Brand"].unique())
brand_emb_dict = {}

for b in brands:
    # find any row belonging to this brand
    idx = data.index[data["Brand"] == b][0]
    brand_emb_dict[b] = brand_embs[idx] / np.linalg.norm(brand_embs[idx])

mean_brand_emb = brand_embs.mean(axis=0)
mean_brand_emb = mean_brand_emb / np.linalg.norm(mean_brand_emb)

# Query

In [13]:
def expand_gender_vec(gv, dim=384):
    # Convert 3-dim gender vector → same dimensionality as BERT embedding
    return np.pad(gv, (0, dim - len(gv)), 'constant')

def detect_brand_in_query(text, brands):
    """ Detect if any brand from the list is mentioned in the text.
        Return the brand if found, else None."""
    t = text.lower()
    for b in brands:
        if b.lower() in t:
            return b
    return None


def detect_gender_in_query(text):
    """Detech gender from the text.
       Return a 2D numpy array representing the gender embedding."""
    t = text.lower()
    if "unisex" in t:
        return np.array([1/np.sqrt(2), 1/np.sqrt(2)])
    if "men" in t or "male" in t:
        return np.array([1.0, 0.0])
    if "women" in t or "female" in t:
        return np.array([0.0, 1.0])
    # default neutral
    return np.array([1/np.sqrt(2), 1/np.sqrt(2)])

def clean_query_text(text, brands):
    """Remove known brand names & gender words so BERT focuses on scent description."""
    t = text.lower()
    for b in brands:
        t = t.replace(b.lower(), "")
    t = t.replace("unisex", "")
    t = t.replace("men", "")
    t = t.replace("man", "")
    t = t.replace("male", "")
    t = t.replace("women", "")
    t = t.replace("woman", "")
    t = t.replace("female", "")
    return t.strip()


def clean_query_text(text, brands):
    """Remove known brand names & gender words so BERT focuses on scent description."""
    t = text.lower()
    for b in brands:
        t = t.replace(b.lower(), "")
    t = t.replace("unisex", "")
    t = t.replace("men", "") 
    t = t.replace("man", "")
    t = t.replace("male", "")
    t = t.replace("women", "")
    t = t.replace("woman", "")
    t = t.replace("female", "")
    return t.strip()

def embed_query(text, model, brands, mean_brand_emb):
    # ----- BRAND -----
    brand = detect_brand_in_query(text, brands)
    if brand:
        brand_emb = brand_emb_dict[brand]     # PCA-reduced or SBERT brand embedding
    else:
        brand_emb = mean_brand_emb            # neutral brand

    # ----- GENDER -----
    gender_vec = detect_gender_in_query(text)
    gender_emb = expand_gender_vec(gender_vec)   # your 2-dim → 384-dim function

    # ----- CLEAN TEXT FOR DESCRIPTION + ACCORD -----
    cleaned = clean_query_text(text, brands)

    # ----- SBERT DESCRIPTION + ACCORD ENCODING -----
    desc_emb   = model.encode([cleaned])[0]
    accord_emb = model.encode([cleaned])[0]
    
    desc_embs = normalize(desc_emb)        # MiniLM output
    brand_embs = normalize(brand_emb)      # PCA brand embeddings
    gender_embs = normalize(gender_emb)
    accord_embs = normalize(accord_emb)


    # ----- FUSION -----
    q = (
        W_DESC   * desc_emb +
        W_ACCORD * accord_emb +
        W_GENDER * gender_emb +
        W_BRAND  * brand_emb
    )

    q = normalize(q)
    return q.astype(np.float32)

def recommend(query, model=model, brands=brands, mean_brand_emb=mean_brand_emb, faiss_index=index, data=data, top_k=10):
    q_emb = embed_query(query, model, brands, mean_brand_emb)
    
    D, I = faiss_index.search(q_emb.reshape(1, -1), top_k)
    
    results = []
    for idx, dist in zip(I[0], D[0]):
        row = data.iloc[idx]
        results.append({
            "Name": row["Name"],
            "Brand": row["Brand"],
            "Gender": row["Gender"],
            "Main Accords": row["Main Accords"],
            "Description": row["Description"],
            "Main Accords": raw_data["Main Accords"][idx],
            "Score": float(dist)
        })
    results_df = pd.DataFrame(results)
    results_df["Main Accords"] = results_df["Main Accords"].apply(lambda x: x.replace("[]", "Not Specified"))
    results_df["Main Accords"] = results_df["Main Accords"].apply(lambda x: x.replace("]", "").replace("[", "").replace("'", "").replace(",", ", "))
    return results_df, results

In [14]:
out, _ = recommend(query="fresh citrus green summer unisex dior")

ValueError: Expected 2D array, got 1D array instead:
array=[-1.24681391e-01  3.57099362e-02 -2.21137516e-02 -5.56254648e-02
 -7.25670978e-02 -5.54278167e-03  3.91429141e-02  2.55377777e-02
 -4.95412201e-03  2.55665556e-02 -1.28395008e-02 -1.64998509e-02
 -2.86132470e-02 -2.54801512e-02 -6.59035668e-02 -9.42420289e-02
 -1.19671538e-01  5.86138517e-02 -2.69689132e-03 -3.60246152e-02
 -2.39398442e-02  5.02062030e-02 -5.30377179e-02  1.13202445e-03
 -3.82177867e-02 -3.49075831e-02  1.24318423e-02 -3.40506597e-03
  4.97921146e-02 -1.04130670e-01  9.44013745e-02  3.88525128e-02
 -1.23312473e-02 -2.42233253e-03 -1.57542974e-02  4.09249701e-02
 -3.52850072e-02 -6.72928691e-02 -8.78986716e-02  2.80129369e-02
 -2.12927703e-02 -2.34899241e-02 -1.05634898e-01  4.67930688e-03
  3.45762111e-02 -9.96523187e-04 -4.51831706e-03  3.14995013e-02
  3.01333740e-02  6.79335669e-02 -2.19647307e-02 -3.16691920e-02
 -4.44952771e-02  8.37735459e-03 -3.32830623e-02  4.02114633e-03
  1.32939145e-02 -1.88957583e-0

In [ ]:
out

,Name,Brand,Gender,Main Accords,Description,Score
0,Oud Caravan No 3 Abdes Salaam Attars,Perfumes,unisex,"oud, fresh spicy",Oud Caravan No 3byAbdes Salaam Attars Perfumes...,0.516041
1,Vetyver Bastide Aix en,Provence,unisex,Not Specified,VetyverbyBastide Aix en Provenceis a Woody fra...,0.487143
2,#455 Tabac écarlate Le Ré,Noir,unisex,Not Specified,#455 Tabac écarlatebyLe Ré Noiris a Amber Spic...,0.483210
3,Aquilae - Oud du Désert Diane,Castel,unisex,"oud, fresh spicy",Aquilae - Oud du DésertbyDiane Castelis a frag...,0.473504
4,Esme L.T.,Piver,unisex,Not Specified,EsmebyL.T. Piveris a fragrance for women and m...,0.461535
5,Illusive Thorn The Dua,Brand,unisex,"oud, fresh spicy",Illusive ThornbyThe Dua Brandis a Amber fragra...,0.457381
6,Oud Té Fueguia,1833,unisex,"oud, green, fresh, fresh spicy",Oud TébyFueguia 1833is a fragrance for women a...,0.456458
7,EO Black Ensar,Oud,unisex,"oud, fresh spicy",EO BlackbyEnsar Oudis a fragrance for women an...,0.454804
8,#211 Mon Vetiver Narcotique Le Ré,Noir,unisex,"aromatic, fruity, green, woody, cannabis, ...",#211 Mon Vetiver NarcotiquebyLe Ré Noiris a fr...,0.454687
9,Oud Caravan No 2 Abdes Salaam Attars,Perfumes,unisex,"oud, white floral, woody, animalic",Oud Caravan No 2byAbdes Salaam Attars Perfumes...,0.454087


In [ ]:
out['Description'][0]

'Oud Caravan No 3byAbdes Salaam Attars Perfumesis a Amber fragrance for women and men. The nose behind this fragrance is Dominique Dubrana.'